# Judge period strings with Claude

Sends each `data/judge-input/batch-*.jsonl` batch which hasn't been processed yet to Claude through the
Message Batches API and saves the reply as `data/judge-output/<batch>.jsonl` for `period_parsers.ipynb`.
The system prompt and the per-batch instruction come from `period_judge_prompt.md`.

In [5]:
import getpass
import json
import os
import time
from pathlib import Path

import anthropic

MODEL = "claude-sonnet-5"
INPUT_DIR = Path("data/judge-input")
OUTPUT_DIR = Path("data/judge-output")
LIMIT = 100 # Number of batches to submit

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key: "))

Anthropic API key:  ········


In [ ]:
system_text, _, user_part = Path("period_judge_prompt.md").read_text(encoding="utf-8").partition("# User message, per batch")
user_template = user_part.split("```")[1].strip().splitlines()[0]  # the instruction line; the batch follows it

system = [{"type": "text", "text": system_text.removeprefix("# System prompt").strip(), "cache_control": {"type": "ephemeral"}}]

In [31]:
pending = [f for f in sorted(INPUT_DIR.glob("batch-*.jsonl")) if not (OUTPUT_DIR / f.name).exists()][:LIMIT]

requests = []
for f in pending:
    lines = f.read_text(encoding="utf-8").strip().splitlines()
    requests.append({
        "custom_id": f.stem,
        "params": {
            "model": MODEL,
            "max_tokens": 32000,
            "system": system,
            "output_config": {"effort": "low"},
            "messages": [{"role": "user", "content": user_template.format(n=len(lines)) + "\n\n" + "\n".join(lines)}],
        },
    })

batch = client.messages.batches.create(requests=requests)
print(f"submitted {len(requests)} requests as batch {batch.id}")

submitted 100 requests as batch msgbatch_0124YwD7e1h2QjMRwUDUzAFc


In [8]:

while (batch := client.messages.batches.retrieve("msgbatch_0124YwD7e1h2QjMRwUDUzAFc")).processing_status != "ended":
    print(f"{batch.request_counts.processing} still processing")
    time.sleep(60)
print(batch.request_counts)

MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=100)


In [9]:
OUTPUT_DIR.mkdir(exist_ok=True)
usage = {"input": 0, "cached": 0, "output": 0}
for result in client.messages.batches.results(batch.id):
    if result.result.type != "succeeded":
        print(f"{result.custom_id}: {result.result.type}")
        continue
    message = result.result.message
    if message.stop_reason != "end_turn":
        print(f"{result.custom_id}: {message.stop_reason}, not saved")
        continue
    text = "".join(block.text for block in message.content if block.type == "text")
    lines = [line for line in text.strip().splitlines() if not line.startswith("```")]
    (OUTPUT_DIR / f"{result.custom_id}.jsonl").write_text("\n".join(lines) + "\n", encoding="utf-8")
    usage["input"] += message.usage.input_tokens
    usage["cached"] += message.usage.cache_read_input_tokens or 0
    usage["output"] += message.usage.output_tokens

print(f"{len(list(OUTPUT_DIR.glob('batch-*.jsonl')))} batches now judged; tokens this run: {usage}")

100 batches now judged; tokens this run: {'input': 358868, 'cached': 403880, 'output': 817655}
